In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [2]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [3]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "effiecientnet_b0",
    "learning_rate": 0.001,
    "epochs": 20,
    "pretrained":True
}

In [4]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ]),

    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],
                             [0.229,0.224,0.225])
    ])
}


test_dir = "../BanglaLekha_8fold/test"

train_dataloaders = []
val_dataloaders = []

for i in range(1, 9):
    train_dir = f"../BanglaLekha_8fold/fold_{i}/train"
    val_dir = f"../BanglaLekha_8fold/fold_{i}/validation"
    train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
    val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])
    train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)

    train_dataloaders.append(train_dataloader)
    val_dataloaders.append(val_dataloader)

test_dataset = datasets.ImageFolder(root=test_dir, transform=data_transforms["test"])
test_dataloader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [6]:

efficientnet_b0 = models.efficientnet_b0(pretrained=True)


for param in efficientnet_b0.parameters():
    param.requires_grad = False

in_features = efficientnet_b0.classifier[1].in_features
efficientnet_b0.classifier[1] = nn.Linear(in_features, 84)

total_params = sum(p.numel() for p in efficientnet_b0.parameters())

gpu = torch.device("cuda")
efficientnet_b0 = efficientnet_b0.to(gpu)


In [7]:
print(total_params)

4115152


In [13]:
import fine_tuning as ft
ft.fine_tune(model=efficientnet_b0, model_name="efficientnet_b0", state="full") #Change the state for fine-tuning

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW([
    {"params": efficientnet_b0.classifier[1].parameters(), "lr": 1e-4, "weight-decay": 1e-4},
    {"params": efficientnet_b0.features.parameters(), "lr": 1e-5, "weight-decay": 1e-4},
    
])
epochs = model_config["epochs"]

In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [11]:
def train_model(model, train_dataloader, val_dataloader, optimizer, criterion, epochs, fold):
    
    max_val_accuracy = 0
    count = 0
    patience = 5

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()
        
        val_accuracy = validate_model(efficientnet_b0, val_dataloader)

        if(val_accuracy > max_val_accuracy):
            max_val_accuracy = val_accuracy
            count = 0

            torch.save(model.state_dict(), f"saved_parameters/efficientnet_b0/efficientnet_b0_fold_{fold}.pth")

        else:
            count = count + 1

        if(count >= patience):
            break

        
        
        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}, Validation Accuracy: {val_accuracy}")

In [14]:
for i in range(0, 8):
    print(f"Fold: {i+1}")
    train_model(efficientnet_b0, train_dataloaders[i], val_dataloaders[i], optimizer, criterion, model_config["epochs"], i+1)

Fold: 1
Epoch: 1, Training Loss: 0.15949505938257647, Validation Accuracy: 0.9740126620440157
Epoch: 2, Training Loss: 0.1348793073784352, Validation Accuracy: 0.9732288212239976
Epoch: 3, Training Loss: 0.11956717310219198, Validation Accuracy: 0.9702743442870063
Epoch: 4, Training Loss: 0.10733469403348543, Validation Accuracy: 0.9699125716008441
Epoch: 5, Training Loss: 0.09783743942614538, Validation Accuracy: 0.9676816400361773
Fold: 2
Epoch: 1, Training Loss: 0.09513244302674846, Validation Accuracy: 0.9931263189629183
Epoch: 2, Training Loss: 0.08712288799138795, Validation Accuracy: 0.9907145010551703
Epoch: 3, Training Loss: 0.07921553073958021, Validation Accuracy: 0.989388001205909
Epoch: 4, Training Loss: 0.0716533357388316, Validation Accuracy: 0.9886041603858908
Epoch: 5, Training Loss: 0.06641019402341249, Validation Accuracy: 0.987579137775098
Fold: 3
Epoch: 1, Training Loss: 0.06337328624289246, Validation Accuracy: 0.9954175459752789
Epoch: 2, Training Loss: 0.0579213

In [15]:
max_accuracy = 0
for i in range(1, 9):
    efficientnet_b0.load_state_dict(torch.load(f"saved_parameters/efficientnet_b0/efficientnet_b0_fold_{i}.pth"))
    accuracy = validate_model(efficientnet_b0, test_dataloader)
    if(accuracy > max_accuracy):
        max_accuracy = accuracy
print(f"Accuracy: {100 * accuracy}")

Accuracy: 93.71215088427586
